In [8]:
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb
from google import genai

In [9]:
client = genai.Client()

In [3]:
# Vector Embeddings
embedding_model = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1103.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
vector_store = Chroma(
    collection_name="learn_rag",
    embedding_function=embedding_model,
    host="localhost",
    port=8000,
)

In [12]:
# Take user query
user_query = input("Ask Something: ")

# Relevant chunks from vector db
search_results = vector_store.similarity_search(query=user_query)

context = "\n\n\n".join([f"Page Content: {result.page_content}\nPage Number: {result.metadata['page_label']}\nFile Location: {result.metadata['source']}" for result in search_results])

SYSTEM_PROMPT = f"""
You are a helpful AI Assistant who answers user query based on the available context retrieved from a PDF file with page_contents and page number.

you should only ans the user based on the following context and navigate the user to open the right page to know more.

Context:
{context}
"""

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=user_query,
    config={
        "system_instruction": SYSTEM_PROMPT,
    }
)

print(f"Response: {response.text}")

Response: Based on the documents provided, there is no information regarding the BERT architecture. The context focuses on the Transformer model architecture, which includes an encoder-decoder structure, self-attention mechanisms, and residual connections.

To learn more about the architecture discussed in the provided documents, please refer to **pages 2 and 3** of `attention_u_need.pdf`.
